# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassaanSaqib/FlyRankAI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My confirmed lane is **Refresh / Content Opportunity Scoring**.

My baseline rule is:

> Prioritise a page for human review when it still has meaningful search visibility and is either stale, underperforming in CTR compared with pages at a similar search position, or both.

I use three safe inputs:

1. **Visibility:** at least 300 impressions over the observed 90-day window.

2. **Staleness:** at least 180 days since the page was last updated.

3. **CTR opportunity:** CTR is at least 20% below the observed median CTR for pages in the same position bucket.

Pages meeting both the staleness and CTR conditions should rank above pages meeting only one. Search visibility is used to prioritise pages where the possible impact is larger.

The rule can output one reason code per page:

- `stale_and_low_ctr`

- `stale_but_visible`

- `low_ctr_for_position`

- `not_priority`

Before encoding the rule, the code below checks two signals using visible bucket tables with sample sizes:

- **Staleness**, which is linked to refresh-review logic.

- **CTR relative to average position**, which is linked to CTR-fix logic.

The one-word verdicts are generated from the observed tables: **CONFIRMED, OPPOSITE, MIXED, or FALSE**. These checks are directional and do not establish causal effects.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

import json

import subprocess

import numpy as np

import pandas as pd

from IPython.display import display, Markdown

# ------------------------------------------------------------

# 0. Load the repository and the correct starter dataset

# ------------------------------------------------------------

REPO_URL = "https://github.com/HassaanSaqib/FlyRankAI-Internship.git"

REPO_DIR = Path("/content/flyrank_repo")

DATA_PATH = REPO_DIR / "data/raw/content_refresh_anonymized.csv"

if not REPO_DIR.exists():

    subprocess.run(

        ["git", "clone", "-q", REPO_URL, str(REPO_DIR)],

        check=True,

    )

if not DATA_PATH.exists():

    raise FileNotFoundError(

        f"Starter dataset not found at {DATA_PATH}"

    )

df = pd.read_csv(DATA_PATH)

required_columns = {

    "content_id",

    "days_since_last_update",

    "impressions_90d",

    "ctr",

    "avg_position",

    "trend_direction",

}

missing_columns = required_columns.difference(df.columns)

if missing_columns:

    raise KeyError(

        f"Required columns are missing: {sorted(missing_columns)}"

    )

numeric_columns = [

    "days_since_last_update",

    "impressions_90d",

    "ctr",

    "avg_position",

]

for column in numeric_columns:

    df[column] = pd.to_numeric(df[column], errors="coerce")

# trend_direction is used only as a retrospective evaluation label.

# It is NEVER used as an input to the baseline score.

df["is_declining_eval"] = (

    df["trend_direction"]

    .astype(str)

    .str.lower()

    .eq("down")

    .astype(int)

)

print("Dataset loaded successfully.")

print(f"Rows: {len(df):,}")

print(f"Columns: {len(df.columns)}")

print(f"Observed declining-label base rate: {df['is_declining_eval'].mean():.1%}")

# ------------------------------------------------------------

# 1A. Signal check: staleness behind refresh logic

# ------------------------------------------------------------

staleness_order = [

    "0-30",

    "31-90",

    "91-180",

    "181-365",

    "365+",

    "missing",

]

days = df["days_since_last_update"]

df["staleness_bucket"] = np.select(

    [

        days.isna(),

        days <= 30,

        days <= 90,

        days <= 180,

        days <= 365,

        days > 365,

    ],

    [

        "missing",

        "0-30",

        "31-90",

        "91-180",

        "181-365",

        "365+",

    ],

    default="missing",

)

df["staleness_bucket"] = pd.Categorical(

    df["staleness_bucket"],

    categories=staleness_order,

    ordered=True,

)

staleness_table = (

    df.groupby("staleness_bucket", observed=False)

    .agg(

        n=("content_id", "size"),

        declining_n=("is_declining_eval", "sum"),

        decline_rate=("is_declining_eval", "mean"),

        median_impressions=("impressions_90d", "median"),

    )

    .reset_index()

)

def staleness_verdict(table):

    valid = table[

        (table["n"] > 0)

        & table["staleness_bucket"].ne("missing")

    ].dropna(subset=["decline_rate"])

    if len(valid) < 3:

        return "MIXED"

    rates = valid["decline_rate"].to_numpy(dtype=float)

    spread = rates.max() - rates.min()

    if spread < 0.03:

        return "FALSE"

    correlation = np.corrcoef(

        np.arange(len(rates)),

        rates,

    )[0, 1]

    change = rates[-1] - rates[0]

    if not np.isfinite(correlation):

        return "MIXED"

    if change >= 0.03 and correlation >= 0.40:

        return "CONFIRMED"

    if change <= -0.03 and correlation <= -0.40:

        return "OPPOSITE"

    return "MIXED"

stale_verdict = staleness_verdict(staleness_table)

display(Markdown("### Signal check 1 — Staleness"))

display(

    staleness_table.style.format(

        {

            "n": "{:,.0f}",

            "declining_n": "{:,.0f}",

            "decline_rate": "{:.1%}",

            "median_impressions": "{:,.0f}",

        }

    )

)

display(Markdown(f"**Verdict: {stale_verdict}**"))

# ------------------------------------------------------------

# 1B. Signal check: CTR versus search position

# ------------------------------------------------------------

position = df["avg_position"]

df["position_bucket"] = np.select(

    [

        position.isna() | position.le(0),

        position.le(3),

        position.le(10),

        position.le(20),

        position.le(50),

        position.gt(50),

    ],

    [

        "no_data",

        "top_3",

        "page_1",

        "striking",

        "page_3_5",

        "deep",

    ],

    default="no_data",

)

position_order = [

    "top_3",

    "page_1",

    "striking",

    "page_3_5",

    "deep",

]

# A volume floor reduces highly unstable CTR values.

position_audit = df[

    (df["impressions_90d"] >= 300)

    & (df["avg_position"] > 0)

].copy()

position_audit["position_bucket"] = pd.Categorical(

    position_audit["position_bucket"],

    categories=position_order,

    ordered=True,

)

position_table = (

    position_audit.groupby("position_bucket", observed=False)

    .agg(

        n=("content_id", "size"),

        median_ctr=("ctr", "median"),

        median_impressions=("impressions_90d", "median"),

    )

    .reset_index()

)

def ctr_position_verdict(table):

    valid = table[

        table["n"] > 0

    ].dropna(subset=["median_ctr"])

    if len(valid) < 3:

        return "MIXED"

    ctr_values = valid["median_ctr"].to_numpy(dtype=float)

    spread = ctr_values.max() - ctr_values.min()

    if spread < 0.05:

        return "FALSE"

    correlation = np.corrcoef(

        np.arange(len(ctr_values)),

        ctr_values,

    )[0, 1]

    if not np.isfinite(correlation):

        return "MIXED"

    if correlation <= -0.40:

        return "CONFIRMED"

    if correlation >= 0.40:

        return "OPPOSITE"

    return "MIXED"

ctr_position_verdict_value = ctr_position_verdict(position_table)

display(Markdown("### Signal check 2 — CTR versus position"))

display(

    position_table.style.format(

        {

            "n": "{:,.0f}",

            "median_ctr": "{:.2f}%",

            "median_impressions": "{:,.0f}",

        }

    )

)

display(

    Markdown(

        f"**Verdict: {ctr_position_verdict_value}**"

    )

)

# Store each position bucket's observed median CTR and sample size.

expected_ctr_map = (

    position_table

    .set_index("position_bucket")["median_ctr"]

    .to_dict()

)

benchmark_n_map = (

    position_table

    .set_index("position_bucket")["n"]

    .to_dict()

)

global_expected_ctr = position_audit["ctr"].median()

df["expected_ctr_for_position"] = pd.to_numeric(

    df["position_bucket"].map(expected_ctr_map),

    errors="coerce",

).fillna(global_expected_ctr)

df["ctr_benchmark_n"] = pd.to_numeric(

    df["position_bucket"].map(benchmark_n_map),

    errors="coerce",

).fillna(0).astype(int)

expected = df["expected_ctr_for_position"].replace(0, np.nan)

df["ctr_gap_ratio"] = (

    (expected - df["ctr"]) / expected

).clip(lower=0, upper=1).fillna(0)

print("\nSignal audit complete.")

print(f"Staleness verdict: {stale_verdict}")

print(f"CTR-vs-position verdict: {ctr_position_verdict_value}")

print(

    "Important: trend_direction was used only as an evaluation outcome, "

    "not as a baseline feature."
)

Dataset loaded successfully.
Rows: 30,000
Columns: 45
Observed declining-label base rate: 54.2%


### Signal check 1 — Staleness

,staleness_bucket,n,declining_n,decline_rate,median_impressions
0,0-30,"20,480","10,473",51.1%,470
1,31-90,175,103,58.9%,510
2,91-180,"9,171","5,604",61.1%,"1,692"
3,181-365,169,79,46.7%,16
4,365+,5,3,60.0%,2
5,missing,0,0,nan%,nan


**Verdict: MIXED**

### Signal check 2 — CTR versus position

,position_bucket,n,median_ctr,median_impressions
0,top_3,507,0.20%,"3,685"
1,page_1,"7,647",0.23%,"3,872"
2,striking,"5,051",0.17%,"1,783"
3,page_3_5,"4,985",0.08%,"1,859"
4,deep,562,0.00%,728


**Verdict: CONFIRMED**


Signal audit complete.
Staleness verdict: MIXED
CTR-vs-position verdict: CONFIRMED
Important: trend_direction was used only as an evaluation outcome, not as a baseline feature.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline is deliberately transparent and contains no fitted model weights.

A page is eligible when it has:

- at least **300 impressions** over the observed 90-day window; and

- valid average-position data.

An eligible page receives a signal for:

- being at least **180 days since its last update**; and/or

- having CTR at least **20% below** the median CTR of pages in the same position bucket.

The raw score multiplies log-scaled visibility by the number of triggered signals. It is then normalised onto a 0–100 scale for readability.

This means:

- High-visibility pages meeting both conditions rank highest.

- Pages meeting one condition may still appear in the queue.

- Pages meeting neither condition receive a score of zero.

The baseline supports review prioritisation only. It does not prove that refreshing a page will cause improved search performance.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
MIN_IMPRESSIONS = 300

STALE_DAYS = 180

CTR_GAP_THRESHOLD = 0.20

df["eligible_for_review"] = (

    (df["impressions_90d"] >= MIN_IMPRESSIONS)

    & (df["avg_position"] > 0)

)

df["stale_flag"] = (

    df["days_since_last_update"]

    .fillna(-1)

    .ge(STALE_DAYS)

)

df["low_ctr_flag"] = (

    df["eligible_for_review"]

    & df["expected_ctr_for_position"].gt(0)

    & df["ctr"].le(

        df["expected_ctr_for_position"]

        * (1 - CTR_GAP_THRESHOLD)

    )

)

df["signal_count"] = (

    df["stale_flag"].astype(int)

    + df["low_ctr_flag"].astype(int)

)

# Visibility affects priority but cannot create a recommendation alone.

impression_cap = max(

    float(df["impressions_90d"].quantile(0.99)),

    1.0,

)

df["visibility_weight"] = (

    np.log1p(df["impressions_90d"].clip(lower=0))

    / np.log1p(impression_cap)

).clip(lower=0, upper=1)

df["raw_baseline_score"] = np.where(

    df["eligible_for_review"],

    df["visibility_weight"] * df["signal_count"],

    0.0,

)

maximum_raw_score = df["raw_baseline_score"].max()

if maximum_raw_score <= 0:

    raise ValueError(

        "The baseline produced no positive scores. "

        "Review the thresholds and signal tables."

    )

df["baseline_action_score"] = (

    100

    * df["raw_baseline_score"]

    / maximum_raw_score

).round(2)

# Exactly one reason-code value is attached to every row.

df["reason_code"] = np.select(

    [

        df["eligible_for_review"]

        & df["stale_flag"]

        & df["low_ctr_flag"],

        df["eligible_for_review"]

        & df["stale_flag"],

        df["eligible_for_review"]

        & df["low_ctr_flag"],

    ],

    [

        "stale_and_low_ctr",

        "stale_but_visible",

        "low_ctr_for_position",

    ],

    default="not_priority",

)

df["action_label"] = np.select(

    [

        df["reason_code"].eq("stale_and_low_ctr"),

        df["reason_code"].eq("stale_but_visible"),

        df["reason_code"].eq("low_ctr_for_position"),

    ],

    [

        "REFRESH_AND_SNIPPET_REVIEW",

        "CONTENT_REFRESH_REVIEW",

        "TITLE_META_REVIEW",

    ],

    default="MONITOR",

)

queue = (

    df.sort_values(

        [

            "baseline_action_score",

            "impressions_90d",

        ],

        ascending=[False, False],

    )

    .reset_index(drop=True)

    .copy()

)

queue["baseline_rank"] = np.arange(1, len(queue) + 1)

# Retrospective evaluation only: not part of the score.

base_rate = float(queue["is_declining_eval"].mean())

precision_at_20 = float(

    queue.head(20)["is_declining_eval"].mean()

)

output_columns = [

    "baseline_rank",

    "content_id",

    "baseline_action_score",

    "reason_code",

    "action_label",

    "impressions_90d",

    "ctr",

    "expected_ctr_for_position",

    "avg_position",

    "days_since_last_update",

    "stale_flag",

    "low_ctr_flag",

]

output_dir = REPO_DIR / "work/outputs"

output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "baseline_action_score.csv"

queue[output_columns].to_csv(

    csv_path,

    index=False,

)

metrics = {

    "lane": "Refresh / Content Opportunity Scoring",

    "row_count": int(len(queue)),

    "minimum_impressions": MIN_IMPRESSIONS,

    "stale_days_threshold": STALE_DAYS,

    "ctr_gap_threshold": CTR_GAP_THRESHOLD,

    "staleness_signal_verdict": stale_verdict,

    "ctr_position_signal_verdict": ctr_position_verdict_value,

    "declining_label_base_rate": base_rate,

    "baseline_precision_at_20": precision_at_20,

    "csv_generated": "work/outputs/baseline_action_score.csv",

}

metrics_path = output_dir / "baseline_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:

    json.dump(metrics, file, indent=2)

print("Ranked queue created successfully.")

print(f"Rows ranked: {len(queue):,}")

print(f"Positive-score pages: {(queue['baseline_action_score'] > 0).sum():,}")

print(f"Observed label base rate: {base_rate:.1%}")

print(f"Baseline Precision@20: {precision_at_20:.1%}")

print(f"CSV written to: {csv_path}")

print(f"Metrics written to: {metrics_path}")

display(

    queue[output_columns]

    .head(20)

    .style.format(

        {

            "baseline_action_score": "{:.2f}",

            "impressions_90d": "{:,.0f}",

            "ctr": "{:.2f}%",

            "expected_ctr_for_position": "{:.2f}%",

            "avg_position": "{:.1f}",

            "days_since_last_update": "{:.0f}",

        }

    )

)

Ranked queue created successfully.
Rows ranked: 30,000
Positive-score pages: 7,710
Observed label base rate: 54.2%
Baseline Precision@20: 60.0%
CSV written to: /content/flyrank_repo/work/outputs/baseline_action_score.csv
Metrics written to: /content/flyrank_repo/work/outputs/baseline_metrics.json


,baseline_rank,content_id,baseline_action_score,reason_code,action_label,impressions_90d,ctr,expected_ctr_for_position,avg_position,days_since_last_update,stale_flag,low_ctr_flag
0,1,content_5feee3994adb,100.00,stale_and_low_ctr,REFRESH_AND_SNIPPET_REVIEW,"7,812",0.01%,0.08%,39.0,194,True,True
1,2,content_b16bd7307b39,94.07,stale_and_low_ctr,REFRESH_AND_SNIPPET_REVIEW,"4,590",0.00%,0.08%,31.0,194,True,True
2,3,content_928af3e22c80,82.97,stale_and_low_ctr,REFRESH_AND_SNIPPET_REVIEW,"1,697",0.12%,0.17%,15.8,193,True,True
3,4,content_074ba6ead17b,70.07,stale_and_low_ctr,REFRESH_AND_SNIPPET_REVIEW,533,0.00%,0.08%,48.0,183,True,True
4,5,content_fd16e3475c29,67.65,stale_and_low_ctr,REFRESH_AND_SNIPPET_REVIEW,429,0.00%,0.23%,9.0,183,True,True
5,6,content_5fe46e04994d,62.50,low_ctr_for_position,TITLE_META_REVIEW,"517,715",0.14%,0.23%,4.2,104,False,True
6,7,content_8c19996aa890,62.50,low_ctr_for_position,TITLE_META_REVIEW,"509,252",0.15%,0.20%,2.5,20,False,True
7,8,content_cb112fce36be,62.50,low_ctr_for_position,TITLE_META_REVIEW,"309,910",0.16%,0.23%,5.6,104,False,True
8,9,content_36ff89c8214e,62.50,low_ctr_for_position,TITLE_META_REVIEW,"295,097",0.05%,0.23%,7.3,104,False,True
9,10,content_b28d1efd668f,62.50,low_ctr_for_position,TITLE_META_REVIEW,"286,608",0.06%,0.08%,26.2,104,False,True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I review the top 20 because the highest-ranked recommendations are where weaknesses in a rule become most visible.

For each page, I record:

- the recommended action;

- the single reason code;

- the measured signals that placed it in the queue;

- a confidence note; and

- a condition that could make the recommendation wrong.

The code creates a first-pass review line for every page. I will read all 20 lines and edit any statement that is not supported by the displayed values. These are review recommendations, not automatic content decisions.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

def build_review(row):

    why_parts = [

        f"{int(row['impressions_90d']):,} impressions",

        f"position {row['avg_position']:.1f}",

    ]

    if bool(row["stale_flag"]):

        update_days = row["days_since_last_update"]

        if pd.notna(update_days):

            why_parts.append(

                f"{int(update_days)} days since update"

            )

    if bool(row["low_ctr_flag"]):

        why_parts.append(

            f"CTR {row['ctr']:.2f}% versus "

            f"{row['expected_ctr_for_position']:.2f}% "

            "position-bucket benchmark"

        )

    if (

        row["signal_count"] == 2

        and row["impressions_90d"] >= 3000

        and row["ctr_benchmark_n"] >= 50

    ):

        confidence = "HIGHER — two signals, material visibility, and a supported CTR benchmark"

    elif row["signal_count"] == 2:

        confidence = "MEDIUM — two signals agree, but context still requires checking"

    else:

        confidence = "CAUTIOUS — only one rule signal triggered"

    reason = row["reason_code"]

    if reason == "stale_and_low_ctr":

        wrong_if = (

            "the page is intentionally seasonal or evergreen, was updated "

            "outside the recorded system, or its low CTR is explained by "

            "query mix or SERP features"

        )

    elif reason == "stale_but_visible":

        wrong_if = (

            "the content remains accurate and evergreen despite its age, "

            "or a refresh would not change the user's search decision"

        )

    elif reason == "low_ctr_for_position":

        wrong_if = (

            "the position-bucket benchmark is not comparable because of "

            "brand intent, query mix, SERP features, or unusually low sample support"

        )

    else:

        wrong_if = (

            "the baseline does not contain enough evidence to recommend action"

        )

    why = "; ".join(why_parts)

    review_line = (

        f"Rank {int(row['baseline_rank'])}: "

        f"{row['action_label']} | "

        f"{reason} | "

        f"Why: {why}. | "

        f"Confidence: {confidence}. | "

        f"Wrong if: {wrong_if}."

    )

    return pd.Series(

        {

            "why_selected": why,

            "confidence_note": confidence,

            "what_would_make_it_wrong": wrong_if,

            "review_line": review_line,

        }

    )

review_details = top20.apply(

    build_review,

    axis=1,

)

top20_review = pd.concat(

    [

        top20.reset_index(drop=True),

        review_details.reset_index(drop=True),

    ],

    axis=1,

)

review_columns = [

    "baseline_rank",

    "content_id",

    "action_label",

    "reason_code",

    "baseline_action_score",

    "why_selected",

    "confidence_note",

    "what_would_make_it_wrong",

]

display(top20_review[review_columns])

print("\nTOP-20 REVIEW LINES\n")

for line in top20_review["review_line"]:

    print(line)

    print()

,baseline_rank,content_id,action_label,reason_code,baseline_action_score,why_selected,confidence_note,what_would_make_it_wrong
0,1,content_5feee3994adb,REFRESH_AND_SNIPPET_REVIEW,stale_and_low_ctr,100.00,"7,812 impressions; position 39.0; 194 days sin...","HIGHER — two signals, material visibility, and...",the page is intentionally seasonal or evergree...
1,2,content_b16bd7307b39,REFRESH_AND_SNIPPET_REVIEW,stale_and_low_ctr,94.07,"4,590 impressions; position 31.0; 194 days sin...","HIGHER — two signals, material visibility, and...",the page is intentionally seasonal or evergree...
2,3,content_928af3e22c80,REFRESH_AND_SNIPPET_REVIEW,stale_and_low_ctr,82.97,"1,697 impressions; position 15.8; 193 days sin...","MEDIUM — two signals agree, but context still ...",the page is intentionally seasonal or evergree...
3,4,content_074ba6ead17b,REFRESH_AND_SNIPPET_REVIEW,stale_and_low_ctr,70.07,533 impressions; position 48.0; 183 days since...,"MEDIUM — two signals agree, but context still ...",the page is intentionally seasonal or evergree...
4,5,content_fd16e3475c29,REFRESH_AND_SNIPPET_REVIEW,stale_and_low_ctr,67.65,429 impressions; position 9.0; 183 days since ...,"MEDIUM — two signals agree, but context still ...",the page is intentionally seasonal or evergree...
5,6,content_5fe46e04994d,TITLE_META_REVIEW,low_ctr_for_position,62.50,"517,715 impressions; position 4.2; CTR 0.14% v...",CAUTIOUS — only one rule signal triggered,the position-bucket benchmark is not comparabl...
6,7,content_8c19996aa890,TITLE_META_REVIEW,low_ctr_for_position,62.50,"509,252 impressions; position 2.5; CTR 0.15% v...",CAUTIOUS — only one rule signal triggered,the position-bucket benchmark is not comparabl...
7,8,content_cb112fce36be,TITLE_META_REVIEW,low_ctr_for_position,62.50,"309,910 impressions; position 5.6; CTR 0.16% v...",CAUTIOUS — only one rule signal triggered,the position-bucket benchmark is not comparabl...
8,9,content_36ff89c8214e,TITLE_META_REVIEW,low_ctr_for_position,62.50,"295,097 impressions; position 7.3; CTR 0.05% v...",CAUTIOUS — only one rule signal triggered,the position-bucket benchmark is not comparabl...
9,10,content_b28d1efd668f,TITLE_META_REVIEW,low_ctr_for_position,62.50,"286,608 impressions; position 26.2; CTR 0.06% ...",CAUTIOUS — only one rule signal triggered,the position-bucket benchmark is not comparabl...



TOP-20 REVIEW LINES

Rank 1: REFRESH_AND_SNIPPET_REVIEW | stale_and_low_ctr | Why: 7,812 impressions; position 39.0; 194 days since update; CTR 0.01% versus 0.08% position-bucket benchmark. | Confidence: HIGHER — two signals, material visibility, and a supported CTR benchmark. | Wrong if: the page is intentionally seasonal or evergreen, was updated outside the recorded system, or its low CTR is explained by query mix or SERP features.

Rank 2: REFRESH_AND_SNIPPET_REVIEW | stale_and_low_ctr | Why: 4,590 impressions; position 31.0; 194 days since update; CTR 0.00% versus 0.08% position-bucket benchmark. | Confidence: HIGHER — two signals, material visibility, and a supported CTR benchmark. | Wrong if: the page is intentionally seasonal or evergreen, was updated outside the recorded system, or its low CTR is explained by query mix or SERP features.

Rank 3: REFRESH_AND_SNIPPET_REVIEW | stale_and_low_ctr | Why: 1,697 impressions; position 15.8; 193 days since update; CTR 0.12% versus 0.17

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

A high score does not guarantee that a recommendation is correct.

I treat a top-20 result as potentially weak when:

- it is supported by only one signal;

- its CTR benchmark is based on a small bucket;

- its average position is too deep for a stable CTR comparison;

- it has limited visibility despite appearing in the top 20; or

- it sits at the edge of the top-20 cutoff.

The baseline score uses only:

- `days_since_last_update`

- `impressions_90d`

- `ctr`

- `avg_position`

It does not use product flags, IDs as predictive inputs, model outputs, future-window fields, `trend_direction`, `trend_pct`, or `is_declining_label`.

`trend_direction` is used only after ranking as a retrospective evaluation outcome. It does not influence the score, reason code, action label, or rank.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------

# 4A. Identify weak or borderline top-20 recommendations

# ------------------------------------------------------------

def weak_pick_reasons(row):

    reasons = []

    if row["signal_count"] < 2:

        reasons.append("only one baseline signal triggered")

    if (

        bool(row["low_ctr_flag"])

        and row["ctr_benchmark_n"] < 50

    ):

        reasons.append(

            f"CTR benchmark has only n={int(row['ctr_benchmark_n'])}"

        )

    if (

        bool(row["low_ctr_flag"])

        and row["avg_position"] > 20

    ):

        reasons.append(

            "CTR comparison is being made at a relatively deep position"

        )

    if row["impressions_90d"] < 1000:

        reasons.append("visibility is limited")

    if pd.isna(row["days_since_last_update"]):

        reasons.append("last-update age is missing")

    return "; ".join(reasons)

top20_review["weak_pick_note"] = top20_review.apply(

    weak_pick_reasons,

    axis=1,

)

weak_picks = top20_review[

    top20_review["weak_pick_note"].ne("")

].copy()

# The baseline skill expects at least one skeptical/borderline pick.

# If no explicit weakness is detected, treat rank 20 as the cutoff case.

if weak_picks.empty:

    cutoff_index = top20_review.index[-1]

    top20_review.loc[

        cutoff_index,

        "weak_pick_note",

    ] = (

        "borderline cutoff pick — a small score or threshold change "

        "could remove it from the top 20"

    )

    weak_picks = top20_review.loc[[cutoff_index]].copy()

display(Markdown("### Weak or borderline picks"))

display(

    weak_picks[

        [

            "baseline_rank",

            "content_id",

            "baseline_action_score",

            "reason_code",

            "impressions_90d",

            "ctr",

            "avg_position",

            "days_since_last_update",

            "weak_pick_note",

        ]

    ]

)

# ------------------------------------------------------------

# 4B. Explicit leakage and reproducibility check

# ------------------------------------------------------------

used_score_features = [

    "days_since_last_update",

    "impressions_90d",

    "ctr",

    "avg_position",

]

forbidden_exact = {

    "trend_direction",

    "trend_pct",

    "is_declining_label",

    "is_declining_eval",

    "final_refresh_score",

    "best_model_name",

    "baseline_action_score",

    "reason_code",

    "action_label",

}

leaked_features = sorted(

    set(used_score_features).intersection(forbidden_exact)

)

future_window_tokens = [

    "future",

    "next_",

    "outcome_",

    "label",

    "trend",

    "final_",

    "model_",

]

suspicious_features = sorted(

    feature

    for feature in used_score_features

    if any(token in feature for token in future_window_tokens)

)

id_features = sorted(

    set(used_score_features)

    .intersection({"content_id", "client_id"})

)

assert not leaked_features, (

    f"Leakage detected: {leaked_features}"

)

assert not suspicious_features, (

    f"Suspicious future/label feature detected: {suspicious_features}"

)

assert not id_features, (

    f"Identifier incorrectly used as a score feature: {id_features}"

)

assert csv_path.exists(), (

    "The ranked CSV was not generated."

)

print("\nLEAKAGE CHECK: PASS")

print("Score inputs:", used_score_features)

print("Label-derived score inputs: none")

print("Future-window score inputs: none")

print("Product/model-output score inputs: none")

print("IDs used as score inputs: none")

print(

    "trend_direction is used only as a retrospective evaluation label."

)

print(f"Ranked CSV exists: {csv_path.exists()}")

print(f"Ranked CSV path: {csv_path}")

### Weak or borderline picks

,baseline_rank,content_id,baseline_action_score,reason_code,impressions_90d,ctr,avg_position,days_since_last_update,weak_pick_note
0,1,content_5feee3994adb,100.00,stale_and_low_ctr,7812,0.01,39.0,194,CTR comparison is being made at a relatively d...
1,2,content_b16bd7307b39,94.07,stale_and_low_ctr,4590,0.00,31.0,194,CTR comparison is being made at a relatively d...
3,4,content_074ba6ead17b,70.07,stale_and_low_ctr,533,0.00,48.0,183,CTR comparison is being made at a relatively d...
4,5,content_fd16e3475c29,67.65,stale_and_low_ctr,429,0.00,9.0,183,visibility is limited
5,6,content_5fe46e04994d,62.50,low_ctr_for_position,517715,0.14,4.2,104,only one baseline signal triggered
6,7,content_8c19996aa890,62.50,low_ctr_for_position,509252,0.15,2.5,20,only one baseline signal triggered
7,8,content_cb112fce36be,62.50,low_ctr_for_position,309910,0.16,5.6,104,only one baseline signal triggered
8,9,content_36ff89c8214e,62.50,low_ctr_for_position,295097,0.05,7.3,104,only one baseline signal triggered
9,10,content_b28d1efd668f,62.50,low_ctr_for_position,286608,0.06,26.2,104,only one baseline signal triggered; CTR compar...
10,11,content_8451fc6f034d,62.50,low_ctr_for_position,272144,0.03,2.3,20,only one baseline signal triggered



LEAKAGE CHECK: PASS
Score inputs: ['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']
Label-derived score inputs: none
Future-window score inputs: none
Product/model-output score inputs: none
IDs used as score inputs: none
trend_direction is used only as a retrospective evaluation label.
Ranked CSV exists: True
Ranked CSV path: /content/flyrank_repo/work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.